In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    split,
    explode,
    trim,
    when,
    monotonically_increasing_id
)

spark = (
    SparkSession.builder
    .appName("Gaming Analytics - Dimensional Model")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.local.dir", "D:/spark_temp")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [23]:
print("\n========== LOAD GOLD DATA ==========\n")

gold = spark.read.parquet(
    "../data/gold/game_analysis"
)

gold.printSchema()

print("Rows:", gold.count())

gold.show(5, truncate=False)


========== LOAD GOLD DATA ==========

root
 |-- app_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- price: double (nullable = true)
 |-- average_playtime: integer (nullable = true)
 |-- positive_ratings: integer (nullable = true)
 |-- negative_ratings: integer (nullable = true)
 |-- total_reviews: long (nullable = true)
 |-- positive_reviews: long (nullable = true)
 |-- negative_reviews: long (nullable = true)
 |-- recommendation_rate: double (nullable = true)
 |-- average_review_score: double (nullable = true)
 |-- rawg_rating: double (nullable = true)
 |-- rawg_metacritic: integer (nullable = true)

Rows: 27075
+------+------------------------+--------------+--------------+----------------------------+-----+----------------+----------------+----------------+-------------+----------------+----------------+-------------------+------------------

In [24]:
print("\n========== CREATE DIM GAME ==========\n")


dim_game = (
    gold
    .select(
        "app_id",
        "name",
        "developer",
        "publisher",
        "genres"
    )
    .dropDuplicates(["app_id"])
)


window = Window.orderBy("app_id")

dim_game = dim_game.withColumn(
    "game_key",
    row_number().over(window)
)


dim_game = dim_game.select(
    "game_key",
    "app_id",
    "name",
    "developer",
    "publisher",
    "genres"
)


dim_game.show(
    5,
    truncate=False
)


========== CREATE DIM GAME ==========

+--------+------+-------------------------+----------------+---------+------+
|game_key|app_id|name                     |developer       |publisher|genres|
+--------+------+-------------------------+----------------+---------+------+
|1       |10    |Counter-Strike           |Valve           |Valve    |Action|
|2       |20    |Team Fortress Classic    |Valve           |Valve    |Action|
|3       |30    |Day of Defeat            |Valve           |Valve    |Action|
|4       |40    |Deathmatch Classic       |Valve           |Valve    |Action|
|5       |50    |Half-Life: Opposing Force|Gearbox Software|Valve    |Action|
+--------+------+-------------------------+----------------+---------+------+
only showing top 5 rows



In [25]:
print("\n========== CREATE PRICE CATEGORY ==========\n")

gold = gold.withColumn(
    "price_category",
    when(col("price") == 0, "Free")
    .when(col("price") <= 10, "Low Price")
    .when(col("price") <= 30, "Medium Price")
    .otherwise("High Price")
)


========== CREATE PRICE CATEGORY ==========



In [26]:
print("\n========== CREATE DIM GENRE ==========\n")

dim_genre = (
    gold
    .select(
        explode(split(col("genres"), ";"))
        .alias("genre")
    )
    .withColumn(
        "genre",
        trim(col("genre"))
    )
    .filter(
        (col("genre").isNotNull()) &
        (col("genre") != "") &
        (col("genre") != "0")
    )
    .dropDuplicates()
    .orderBy("genre")
)

window = Window.orderBy("genre")

dim_genre = dim_genre.withColumn(
    "genre_key",
    row_number().over(window)
)

dim_genre = dim_genre.select(
    "genre_key",
    "genre"
)

dim_genre.show(20, False)


========== CREATE DIM GENRE ==========

+---------+-----------------------+
|genre_key|genre                  |
+---------+-----------------------+
|1        |Accounting             |
|2        |Action                 |
|3        |Adventure              |
|4        |Animation & Modeling   |
|5        |Audio Production       |
|6        |Casual                 |
|7        |Design & Illustration  |
|8        |Documentary            |
|9        |Early Access           |
|10       |Education              |
|11       |Free to Play           |
|12       |Full controller support|
|13       |Game Development       |
|14       |Gore                   |
|15       |Indie                  |
|16       |Massively Multiplayer  |
|17       |Nudity                 |
|18       |Photo Editing          |
|19       |RPG                    |
|20       |Racing                 |
+---------+-----------------------+
only showing top 20 rows



In [27]:
print("\n========== CREATE DIM PUBLISHER ==========\n")

dim_publisher = (
    gold
    .select("publisher")
    .filter(
        (col("publisher").isNotNull()) &
        (trim(col("publisher")) != "")
    )
    .dropDuplicates()
    .orderBy("publisher")
)

window = Window.orderBy("publisher")

dim_publisher = dim_publisher.withColumn(
    "publisher_key",
    row_number().over(window)
)

dim_publisher = dim_publisher.select(
    "publisher_key",
    "publisher"
)

dim_publisher.show(20, False)


========== CREATE DIM PUBLISHER ==========

+-------------+-----------------------------------------+
|publisher_key|publisher                                |
+-------------+-----------------------------------------+
|1            |!Lim studio                              |
|2            |""FieryElsa"";""Ellie"";Adityaraj Rukhane|
|3            |"Atlas ""Incawporated"""                 |
|4            |"Evan ""Solidplasma"" Peiperl"           |
|5            |"Project ""The Game"""                   |
|6            |"Steve ""Khad"" Grant"                   |
|7            |(STCG) Smoker The Car Game               |
|8            |(none)                                   |
|9            |++Good Games;GameChanger Charity         |
|10           |+7 Software                              |
|11           |+Mpact Games, LLC.                       |
|12           |-                                        |
|13           |--                                       |
|14           |-Yodasaurus-

In [28]:
print("\n========== CREATE DIM DEVELOPER ==========\n")

dim_developer = (
    gold
    .select("developer")
    .filter(
        (col("developer").isNotNull()) &
        (trim(col("developer")) != "")
    )
    .dropDuplicates()
    .orderBy("developer")
)

window = Window.orderBy("developer")

dim_developer = dim_developer.withColumn(
    "developer_key",
    row_number().over(window)
)

dim_developer = dim_developer.select(
    "developer_key",
    "developer"
)

dim_developer.show(20, False)


========== CREATE DIM DEVELOPER ==========

+-------------+-----------------------------------------------------------------------------------------------------+
|developer_key|developer                                                                                            |
+-------------+-----------------------------------------------------------------------------------------------------+
|1            |"Andrew ""Versus"" Larionov"                                                                         |
|2            |"Erik Svedäng;El Huervo / Niklas Åkerblad;Tobias Sjögren;Oscar ""Ratvader"" Rydelius;Johannes Gotlén"|
|3            |"Gustav ""Goffa"" Söderström"                                                                        |
|4            |"Imperium42® Game Studio;Dylan Hunt, ""Xblade"";Elsa Hunt                                            |
|5            |"Mario D'Eliso ""Elis-D"""                                                                           |
|6         

In [29]:
print("\n========== CREATE DIM PRICE ==========\n")

dim_price = (
    gold
    .select("price_category")
    .dropDuplicates()
    .orderBy("price_category")
)

window = Window.orderBy("price_category")

dim_price = dim_price.withColumn(
    "price_key",
    row_number().over(window)
)

dim_price = dim_price.select(
    "price_key",
    "price_category"
)

dim_price.show()


========== CREATE DIM PRICE ==========

+---------+--------------+
|price_key|price_category|
+---------+--------------+
|        1|          Free|
|        2|    High Price|
|        3|     Low Price|
|        4|  Medium Price|
+---------+--------------+



In [30]:
print("\n========== PREPARE FACT GENRE ==========\n")

fact = gold.withColumn(
    "genre",
    explode(
        split(
            col("genres"),
            ";"
        )
    )
)

fact = fact.withColumn(
    "genre",
    trim(col("genre"))
)

fact = fact.filter(
    (col("genre").isNotNull()) &
    (col("genre") != "") &
    (col("genre") != "0")
)


========== PREPARE FACT GENRE ==========



In [31]:
fact = fact.join(
    dim_genre,
    "genre",
    "left"
)

In [32]:
fact = fact.join(
    dim_publisher,
    "publisher",
    "left"
)

In [33]:
fact = fact.join(
    dim_developer,
    "developer",
    "left"
)

In [34]:
fact = fact.join(
    dim_price,
    "price_category",
    "left"
)

In [35]:
print("\n========== ADD GAME KEY ==========\n")


fact = fact.join(
    dim_game.select(
        "app_id",
        "game_key"
    ),
    "app_id",
    "left"
)


========== ADD GAME KEY ==========



In [36]:
fact = fact.select(
    "game_key",
    "app_id",
    
    "genre_key",
    "publisher_key",
    "developer_key",
    "price_key",

    "price",
    "average_playtime",
    "positive_ratings",
    "negative_ratings",

    "total_reviews",
    "positive_reviews",
    "negative_reviews",

    "recommendation_rate",
    "average_review_score",

    "rawg_rating",
    "rawg_metacritic"
)

fact.show(10, False)

+--------+------+---------+-------------+-------------+---------+-----+----------------+----------------+----------------+-------------+----------------+----------------+-------------------+--------------------+-----------+---------------+
|game_key|app_id|genre_key|publisher_key|developer_key|price_key|price|average_playtime|positive_ratings|negative_ratings|total_reviews|positive_reviews|negative_reviews|recommendation_rate|average_review_score|rawg_rating|rawg_metacritic|
+--------+------+---------+-------------+-------------+---------+-----+----------------+----------------+----------------+-------------+----------------+----------------+-------------------+--------------------+-----------+---------------+
|1       |10    |2        |12532        |15119        |3        |7.19 |17612           |124534          |3339            |10231        |9886            |0               |0.966              |0.9325579122275437  |4.06       |88             |
|17      |380   |2        |12532        

In [37]:
print("\n========== SAVE DIMENSION TABLES ==========\n")

fact.write.mode("overwrite").parquet(
    "../data/gold/fact_game_analysis"
)

dim_game.write.mode("overwrite").parquet(
    "../data/gold/dim_game"
)

dim_genre.write.mode("overwrite").parquet(
    "../data/gold/dim_genre"
)

dim_publisher.write.mode("overwrite").parquet(
    "../data/gold/dim_publisher"
)

dim_developer.write.mode("overwrite").parquet(
    "../data/gold/dim_developer"
)

dim_price.write.mode("overwrite").parquet(
    "../data/gold/dim_price"
)

print("All tables saved.")


========== SAVE DIMENSION TABLES ==========

All tables saved.


In [38]:
print("\n========== VERIFY TABLES ==========\n")

tables = [
    "fact_game_analysis",
    "dim_game",
    "dim_genre",
    "dim_publisher",
    "dim_developer",
    "dim_price"
]

for table in tables:

    print("\n", "="*60)
    print(table)

    df = spark.read.parquet(f"../data/gold/{table}")

    print("Rows:", df.count())

    df.printSchema()

    df.show(5, truncate=False)


========== VERIFY TABLES ==========


fact_game_analysis
Rows: 76462
root
 |-- game_key: integer (nullable = true)
 |-- app_id: integer (nullable = true)
 |-- genre_key: integer (nullable = true)
 |-- publisher_key: integer (nullable = true)
 |-- developer_key: integer (nullable = true)
 |-- price_key: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- average_playtime: integer (nullable = true)
 |-- positive_ratings: integer (nullable = true)
 |-- negative_ratings: integer (nullable = true)
 |-- total_reviews: long (nullable = true)
 |-- positive_reviews: long (nullable = true)
 |-- negative_reviews: long (nullable = true)
 |-- recommendation_rate: double (nullable = true)
 |-- average_review_score: double (nullable = true)
 |-- rawg_rating: double (nullable = true)
 |-- rawg_metacritic: integer (nullable = true)

+--------+------+---------+-------------+-------------+---------+-----+----------------+----------------+----------------+-------------+----------------+--

In [39]:
paths = [
    "../data/gold/game_analysis",
    "../data/gold/fact_game_analysis",
    "../data/gold/dim_game",
    "../data/gold/dim_genre",
    "../data/gold/dim_publisher",
    "../data/gold/dim_developer",
    "../data/gold/dim_price"
]


for path in paths:

    print("\n")
    print("="*70)
    print(path)
    print("="*70)

    df = spark.read.parquet(path)

    print("Rows:", df.count())

    df.printSchema()

    df.show(
        3,
        truncate=False
    )



../data/gold/game_analysis
Rows: 27075
root
 |-- app_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- price: double (nullable = true)
 |-- average_playtime: integer (nullable = true)
 |-- positive_ratings: integer (nullable = true)
 |-- negative_ratings: integer (nullable = true)
 |-- total_reviews: long (nullable = true)
 |-- positive_reviews: long (nullable = true)
 |-- negative_reviews: long (nullable = true)
 |-- recommendation_rate: double (nullable = true)
 |-- average_review_score: double (nullable = true)
 |-- rawg_rating: double (nullable = true)
 |-- rawg_metacritic: integer (nullable = true)

+------+------------------------+---------+---------+----------------------------+-----+----------------+----------------+----------------+-------------+----------------+----------------+-------------------+--------------------+-----------+-----

In [40]:
from pyspark.sql.functions import col, count, when


fact = spark.read.parquet(
    "../data/gold/fact_game_analysis"
)


fact.select(
    count("*").alias("total_games"),
    count(
        when(
            col("rawg_rating").isNotNull(),
            True
        )
    ).alias("rawg_rating_available"),
    count(
        when(
            col("rawg_metacritic").isNotNull(),
            True
        )
    ).alias("metacritic_available")
).show()

+-----------+---------------------+--------------------+
|total_games|rawg_rating_available|metacritic_available|
+-----------+---------------------+--------------------+
|      76462|                 1160|                 990|
+-----------+---------------------+--------------------+

